In [ ]:
# ============================================================
# PHASE 3 — EXPLORATION & ANALYSE STATISTIQUE (EDA)
# Dataset : Olist Brazilian E-Commerce
# ============================================================

# ── 0. IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# ── 1. CHARGEMENT DES DONNÉES (issues de Phase 2) ───────────
orders        = pd.read_csv('../data/raw/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date'])
order_items   = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
order_payments= pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
customers     = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products      = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers       = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_tr   = pd.read_csv('../data/raw/product_category_name_translation.csv')

# ── 2. JOINTURE MASTER TABLE ─────────────────────────────────
df = (orders
      .merge(order_items,    on='order_id',        how='left')
      .merge(order_payments, on='order_id',        how='left')
      .merge(order_reviews,  on='order_id',        how='left')
      .merge(customers,      on='customer_id',     how='left')
      .merge(products,       on='product_id',      how='left')
      .merge(sellers,        on='seller_id',       how='left')
      .merge(category_tr,    on='product_category_name', how='left')
)

print(f"Shape master table : {df.shape}")
df.head(3)

In [ ]:
# ── 3. FEATURE ENGINEERING PRÉ-EDA ──────────────────────────

# Délai de livraison réel vs estimé
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delay_days']    = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days  # positif = en retard
df['is_late']       = (df['delay_days'] > 0).astype(int)

# Mois & jour de commande
df['order_month']   = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_dow']     = df['order_purchase_timestamp'].dt.day_name()
df['order_hour']    = df['order_purchase_timestamp'].dt.hour

# Revenue par commande
df['revenue'] = df['price'] + df['freight_value']

print("Features créées : delivery_days, delay_days, is_late, order_month, order_dow, order_hour, revenue")

In [ ]:
# ── 4. ANALYSE DES DISTRIBUTIONS ────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Distributions des variables clés", fontsize=16, fontweight='bold')

# 4.1 Prix
axes[0,0].hist(df['price'].dropna(), bins=80, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribution du Prix (BRL)')
axes[0,0].set_xlabel('Prix'); axes[0,0].set_ylabel('Fréquence')

# 4.2 Score avis
review_counts = df['review_score'].value_counts().sort_index()
axes[0,1].bar(review_counts.index, review_counts.values, color='coral')
axes[0,1].set_title('Distribution des Scores Avis')
axes[0,1].set_xlabel('Score'); axes[0,1].set_ylabel('Nombre')

# 4.3 Délai livraison
axes[0,2].hist(df['delivery_days'].dropna(), bins=60, color='mediumseagreen', edgecolor='white')
axes[0,2].set_title('Délai de Livraison (jours)')
axes[0,2].set_xlabel('Jours')

# 4.4 Freight value
axes[1,0].hist(df['freight_value'].dropna(), bins=60, color='mediumpurple', edgecolor='white')
axes[1,0].set_title('Distribution du Frais de Port (BRL)')
axes[1,0].set_xlabel('Frais de Port')

# 4.5 Retard (delay_days)
axes[1,1].hist(df['delay_days'].dropna().clip(-30, 60), bins=80, color='tomato', edgecolor='white')
axes[1,1].axvline(0, color='black', linestyle='--', label='Livraison à temps')
axes[1,1].set_title('Retard vs Estimation (jours)')
axes[1,1].set_xlabel('Jours de retard (positif = en retard)')
axes[1,1].legend()

# 4.6 Nombre d'items par commande
items_per_order = df.groupby('order_id')['order_item_id'].max()
axes[1,2].hist(items_per_order, bins=30, color='goldenrod', edgecolor='white')
axes[1,2].set_title("Nombre d'items par commande")

plt.tight_layout()
plt.savefig('../reports/figures/phase3_distributions.png', dpi=150)
plt.show()

In [ ]:
# ── 5. DÉTECTION DES OUTLIERS (Boxplots & IQR) ──────────────

num_cols = ['price', 'freight_value', 'delivery_days', 'delay_days', 'payment_value']

fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 6))
fig.suptitle("Boxplots — Détection des Outliers", fontsize=14, fontweight='bold')

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    axes[i].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col)
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    axes[i].set_xlabel(f'{n_outliers} outliers IQR')

plt.tight_layout()
plt.savefig('../reports/figures/phase3_boxplots.png', dpi=150)
plt.show()

# Traitement : on cap les outliers au 99e percentile pour les analyses suivantes
for col in ['price', 'freight_value', 'payment_value']:
    cap = df[col].quantile(0.99)
    df[f'{col}_capped'] = df[col].clip(upper=cap)
    print(f"{col} : cap au 99e pctile = {cap:.2f} BRL")

In [ ]:
# ── 6. ANALYSE DE CORRÉLATION ────────────────────────────────

corr_cols = ['price_capped', 'freight_value_capped', 'payment_value_capped',
             'delivery_days', 'delay_days', 'review_score', 'is_late']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title("Heatmap de Corrélation — Variables Numériques", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/phase3_correlation_heatmap.png', dpi=150)
plt.show()

print("\nTop corrélations avec review_score :")
print(corr_matrix['review_score'].sort_values())

In [ ]:
# ── 7. VIOLIN PLOTS — Score Avis vs Retard ───────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Relation Livraison & Satisfaction Client", fontsize=14, fontweight='bold')

# 7.1 Violin : score vs is_late
df_sample = df[['review_score', 'is_late']].dropna()
df_sample['Livraison'] = df_sample['is_late'].map({0: 'À temps', 1: 'En retard'})
sns.violinplot(data=df_sample, x='Livraison', y='review_score',
               palette=['mediumseagreen', 'tomato'], ax=axes[0])
axes[0].set_title('Score Avis : À temps vs En retard')
axes[0].set_ylabel('Score (1–5)')

# 7.2 Box : delivery_days par score
df_clean = df[['review_score', 'delivery_days']].dropna()
df_clean['review_score'] = df_clean['review_score'].astype(int)
sns.boxplot(data=df_clean, x='review_score', y='delivery_days',
            palette='coolwarm', ax=axes[1])
axes[1].set_title('Délai de Livraison par Score Avis')
axes[1].set_xlabel('Score Avis'); axes[1].set_ylabel('Jours de livraison')

plt.tight_layout()
plt.savefig('../reports/figures/phase3_violin_delivery_score.png', dpi=150)
plt.show()

In [ ]:
# ── 8. TESTS STATISTIQUES ────────────────────────────────────

print("="*60)
print("TEST 1 — Mann-Whitney U : Score avis (à temps vs en retard)")
print("="*60)
on_time = df[df['is_late'] == 0]['review_score'].dropna()
late    = df[df['is_late'] == 1]['review_score'].dropna()
stat, p = stats.mannwhitneyu(on_time, late, alternative='greater')
print(f"  U-stat = {stat:.0f},  p-value = {p:.2e}")
print(f"  → {'Différence significative ✅' if p < 0.05 else 'Pas de différence significative'}")
print(f"  Médianes : à temps = {on_time.median():.1f}, en retard = {late.median():.1f}\n")

print("="*60)
print("TEST 2 — ANOVA : Délai de livraison selon le score avis")
print("="*60)
groups = [df[df['review_score'] == s]['delivery_days'].dropna() for s in [1,2,3,4,5]]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"  F-stat = {f_stat:.2f},  p-value = {p_anova:.2e}")
print(f"  → {'Différences significatives entre groupes ✅' if p_anova < 0.05 else 'Pas de différence'}\n")

print("="*60)
print("TEST 3 — Chi-² : Retard vs État du client (top 5 états)")
print("="*60)
top_states = df['customer_state'].value_counts().head(5).index
df_chi = df[df['customer_state'].isin(top_states)][['customer_state','is_late']].dropna()
contingency = pd.crosstab(df_chi['customer_state'], df_chi['is_late'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f"  Chi² = {chi2:.2f},  p-value = {p_chi:.2e},  ddl = {dof}")
print(f"  → {'Association significative ✅' if p_chi < 0.05 else 'Pas d association'}")

In [ ]:
# ── 9. ANALYSE TEMPORELLE ────────────────────────────────────

# 9.1 Évolution mensuelle du CA
monthly = (df.groupby('order_month')
             .agg(revenue=('revenue','sum'), orders=('order_id','nunique'))
             .reset_index()
             .sort_values('order_month'))

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly['order_month'], monthly['revenue']/1000, color='steelblue', alpha=0.6, label='CA (k BRL)')
ax2.plot(monthly['order_month'], monthly['orders'], color='tomato', linewidth=2, marker='o', label='Nb commandes')
ax1.set_xlabel('Mois'); ax1.set_ylabel('CA (k BRL)', color='steelblue')
ax2.set_ylabel('Nb Commandes', color='tomato')
plt.title("Évolution Mensuelle — CA & Volume Commandes", fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.savefig('../reports/figures/phase3_monthly_trend.png', dpi=150)
plt.show()

# 9.2 Heatmap commandes par jour/heure
heatmap_data = df.groupby(['order_dow', 'order_hour'])['order_id'].count().unstack()
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = heatmap_data.reindex(days_order)

plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.5,
            cbar_kws={"label": "Nb Commandes"})
plt.title("Heatmap — Volume Commandes par Jour & Heure", fontsize=14, fontweight='bold')
plt.xlabel('Heure'); plt.ylabel('Jour')
plt.tight_layout()
plt.savefig('../reports/figures/phase3_heatmap_dow_hour.png', dpi=150)
plt.show()

In [ ]:
# ── 10. TOP CATÉGORIES ───────────────────────────────────────

cat_perf = (df.groupby('product_category_name_english')
              .agg(
                  revenue=('revenue','sum'),
                  orders=('order_id','nunique'),
                  avg_score=('review_score','mean'),
                  late_rate=('is_late','mean')
              )
              .dropna()
              .sort_values('revenue', ascending=False)
              .head(15)
              .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Revenu par catégorie
axes[0].barh(cat_perf['product_category_name_english'], cat_perf['revenue']/1000, color='steelblue')
axes[0].set_xlabel('Revenue (k BRL)'); axes[0].set_title('Top 15 Catégories — Revenue')
axes[0].invert_yaxis()

# Score moyen vs taux de retard (scatter)
sc = axes[1].scatter(cat_perf['late_rate']*100, cat_perf['avg_score'],
                     s=cat_perf['revenue']/5000, c=cat_perf['revenue'],
                     cmap='YlOrRd', alpha=0.8, edgecolors='gray')
for _, row in cat_perf.iterrows():
    axes[1].annotate(row['product_category_name_english'][:15],
                     (row['late_rate']*100, row['avg_score']), fontsize=7)
axes[1].set_xlabel('Taux de Retard (%)'); axes[1].set_ylabel('Score Avis Moyen')
axes[1].set_title('Score vs Retard par Catégorie (taille = Revenue)')
plt.colorbar(sc, ax=axes[1], label='Revenue (BRL)')

plt.tight_layout()
plt.savefig('../reports/figures/phase3_category_analysis.png', dpi=150)
plt.show()

In [ ]:
# ── 11. SEGMENTATION CLIENT — RFM ───────────────────────────

snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = (df.groupby('customer_unique_id')
         .agg(
             recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
             frequency=('order_id', 'nunique'),
             monetary=('revenue', 'sum')
         )
         .reset_index())

print(f"Table RFM : {rfm.shape}")
rfm.describe()

In [ ]:
# ── 12. CLUSTERING K-MEANS ──────────────────────────────────

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Normalisation
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency','frequency','monetary']])

# Elbow method
inertias, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'bo-'); axes[0].set_title('Elbow Curve'); axes[0].set_xlabel('K')
axes[1].plot(list(K_range), sil_scores, 'rs-'); axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('K')
plt.tight_layout()
plt.savefig('../reports/figures/phase3_elbow_silhouette.png', dpi=150)
plt.show()

# Choix optimal
k_opt = list(K_range)[sil_scores.index(max(sil_scores))]
print(f"K optimal : {k_opt}  (silhouette = {max(sil_scores):.3f})")

In [ ]:
# K optimal retenu
km_final = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(rfm_scaled)

# Profil des clusters
cluster_profile = rfm.groupby('cluster')[['recency','frequency','monetary']].mean().round(2)
cluster_profile['size'] = rfm.groupby('cluster').size()
cluster_profile['pct']  = (cluster_profile['size'] / len(rfm) * 100).round(1)

# Labels métier
labels_map = {
    cluster_profile['monetary'].idxmax(): '🏆 Champions',
    cluster_profile['recency'].idxmax():  '💤 Dormants',
    cluster_profile['frequency'].idxmax(): '🔄 Fidèles',
}
cluster_profile['label'] = cluster_profile.index.map(lambda i: labels_map.get(i, f'Segment {i}'))
print(cluster_profile)

# Scatter 3D simplifié (2 vues)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.Set1(np.linspace(0, 1, k_opt))

for cluster_id in range(k_opt):
    mask = rfm['cluster'] == cluster_id
    label = cluster_profile.loc[cluster_id, 'label']
    axes[0].scatter(rfm[mask]['recency'], rfm[mask]['monetary'],
                    alpha=0.4, s=5, color=colors[cluster_id], label=label)
    axes[1].scatter(rfm[mask]['frequency'], rfm[mask]['monetary'],
                    alpha=0.4, s=5, color=colors[cluster_id], label=label)

axes[0].set_xlabel('Recency (jours)'); axes[0].set_ylabel('Monetary (BRL)')
axes[0].set_title('Recency vs Monetary'); axes[0].legend(markerscale=3)
axes[1].set_xlabel('Frequency'); axes[1].set_ylabel('Monetary (BRL)')
axes[1].set_title('Frequency vs Monetary'); axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig('../reports/figures/phase3_clusters_rfm.png', dpi=150)
plt.show()

In [ ]:
# ── 13. DBSCAN — Détection d'anomalies ──────────────────────

from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.5, min_samples=10).fit(rfm_scaled)
rfm['dbscan_label'] = db.labels_

n_anomalies = (rfm['dbscan_label'] == -1).sum()
print(f"DBSCAN — Anomalies détectées : {n_anomalies} clients ({n_anomalies/len(rfm)*100:.1f}%)")

# Profil des anomalies
print("\nProfil des clients anormaux :")
print(rfm[rfm['dbscan_label'] == -1][['recency','frequency','monetary']].describe())

In [ ]:
# ── 14. SCATTER MATRIX ──────────────────────────────────────

sample = rfm.sample(min(3000, len(rfm)), random_state=42)
pd.plotting.scatter_matrix(
    sample[['recency','frequency','monetary']],
    c=sample['cluster'], cmap='Set1',
    alpha=0.3, figsize=(10,10), diagonal='kde'
)
plt.suptitle("Scatter Matrix RFM — Couleurs = Clusters", y=1.02, fontsize=13)
plt.savefig('../reports/figures/phase3_scatter_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 15. EXPORT DES DONNÉES ENRICHIES ────────────────────────

rfm.to_csv('../data/processed/rfm_clusters.csv', index=False)
df.to_csv('../data/processed/master_table.csv', index=False)

print("✅ Phase 3 terminée — fichiers sauvegardés :")
print("  → data/processed/rfm_clusters.csv")
print("  → data/processed/master_table.csv")
print("  → reports/figures/phase3_*.png")